# Full Optimization Pipeline on HSCs.

## Prepare the notebook.

### Flags for autoreloading.

In [ ]:
AUTORELOAD = True
IS_EXT_LOADED = False

### Optionally autoreload the notebook.

In [ ]:
if AUTORELOAD:
    if not IS_EXT_LOADED:
        %load_ext autoreload
        IS_EXT_LOADED = True
    assert IS_EXT_LOADED
    %autoreload 2

### Import libraries.

In [ ]:
from collections import defaultdict
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial.distance import cdist
from scipy.special import softmax
from scipy.stats import gaussian_kde
from sklearn.neighbors import KernelDensity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import torch
from tqdm import tqdm
import yaml

from sc_exp_design.constants import DataFields, ParamsFields, PredictionFields
from sc_exp_design.config import NeuralVelocityFieldConfig
from sc_exp_design.metrics import compute_e_distance
from sc_exp_design.models import FlowMatching, TargetPredictionModel
from sc_exp_design.utils import set_reproducibility
from sc_exp_design.training.callbacks import MetricsCallBack, TrainingCallBacks


### Define directories.

In [ ]:
BASE_DIR = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC"
PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-12-14_11-53-04/cerulean-durian-19_FlowMatching.pkl"
TARGET_PREDICTION_MODEL_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_21-22-45/dashing-frog-1258_TargetPredictionModel.pkl"
PRIOR_FLOW_CHECKPOINT_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/prior_flow_FlowMatching.pkl"
ANNOTATION_CONFIG_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/generative_modeling/conditional_flow_matching/config/annotation/default.yaml"


### Define constants.

In [ ]:
perturbation_obs_key = "protocol_id"
cell_state_rep = "X_channel_standardized+X_scatter_standardized"

RANDOM_SEED = 42

### Import additional modules.

In [ ]:
sys.path.insert(0, f"{BASE_DIR}/inverse/inverse_utils")
sys.path.insert(0, f"{BASE_DIR}/model_utils")
from forward_model import ForwardModel
from lambda_schedulers import schedulers_dict
from loss_guidance import LossGuidedFlow
from z_norm_modules import ZNorm, IZNorm, RescaledTargetPredictionModel


### Read annotation config.

In [ ]:
with open(ANNOTATION_CONFIG_PATH, "r") as fb:
    annotation_dict = yaml.safe_load(fb)

### Retrieve scatter features identifiers.

In [ ]:
scatter_columns = annotation_dict["scatter_columns"]
n_scatter_feats = len(scatter_columns)
print(f"{n_scatter_feats=}")

### Setting seed for reproducibility.

In [ ]:
set_reproducibility(RANDOM_SEED)

## Load pretrained models.

### Load cellular response prediction model ($\Phi$).

We will use a pre-trained Conditional Flow Matching model to generate synthetic marker expression and morphological features under distinct medium conditions.

In [ ]:
perturbation_response_prediction_model = FlowMatching.load(
    PERTURBATION_RESPONSE_MODEL_CHECKPOINT_PATH
)
print(f"{perturbation_response_prediction_model.velocity_field}")
perturbation_response_prediction_model.trainer.plot_training_logs()

### Load cell type prediction model ($g$).

In [ ]:
target_prediction_model = TargetPredictionModel.load(
    TARGET_PREDICTION_MODEL_CHECKPOINT_PATH
)
# set label encoder, we are going to need it 
ct_le = LabelEncoder().fit(target_prediction_model.train_data.adata.obs["cell_type"].values)
print(f"{target_prediction_model.target_prediction_model}")

target_prediction_model.target_predictor_trainer.plot_training_logs()

### Load prior flow model $u^\theta_t(\mathbb x, \mathbb e)$.

In [ ]:
prior_flow = FlowMatching.load(
    PRIOR_FLOW_CHECKPOINT_PATH
)
perturbation_response_prediction_model.trainer.plot_training_logs()

## Prepare reference data.

### Retrieve data for $\Phi$ surrogate.

In [ ]:
train_adata_phi = perturbation_response_prediction_model.train_data.adata
val_adata_phi = perturbation_response_prediction_model.validation_data[0].adata
print(f"{train_adata_phi=}\n{val_adata_phi=}")

### Retrieve data for $g$ model.

In [ ]:
train_adata_g = target_prediction_model.train_data.adata
val_adata_g = target_prediction_model.validation_data.adata
print(f"{train_adata_g=}\n{val_adata_g=}")

### Subsampling data for $\Phi$ surrogate.

In [ ]:
n_train_cells = 70_000
n_val_cells = 30_000
train_adata_phi = sc.pp.sample(
    train_adata_phi,
    n=n_train_cells,
    copy=True
)
val_adata_phi = sc.pp.sample(
    val_adata_phi,
    n=n_val_cells,
    copy=True
)
print(f"{train_adata_phi=}, {val_adata_phi=}")

### Concatenate data and compute emebeddings.

In [ ]:
adata_phi = sc.concat((train_adata_phi, val_adata_phi), uns_merge="same")
sc.pp.pca(adata_phi)
sc.pp.neighbors(adata_phi)
sc.tl.umap(adata_phi)
adata_phi

## Define rescaling modules.

### Small factory to instantiate rescaling modules.

In [ ]:
def get_rescaling(
    adata: sc.AnnData,
    channel_params_key: str = "X_channel_params", 
    scatter_params_key: str = "X_scatter_params",
    inverse: bool = False,
) -> ZNorm | IZNorm:
    X_channel_params_fwd = adata.uns[channel_params_key]
    X_scatter_params_fwd = adata.uns[scatter_params_key]
    params = {
        ParamsFields.MEAN: torch.from_numpy(
            np.concatenate(
                (
                    X_channel_params_fwd[ParamsFields.MEAN],
                    X_scatter_params_fwd[ParamsFields.MEAN]
                ), axis=0
            )
        ),
        ParamsFields.COVARIANCE: torch.from_numpy(
            np.concatenate(
                (
                    X_channel_params_fwd["std"],
                    X_scatter_params_fwd["std"]
                ), axis=0
            )
        ),
    }
    if inverse:
        return IZNorm(params)
    return ZNorm(params)

### Get Transformations from Cellular response prediction model.

In [ ]:
z_norm_phi = get_rescaling(train_adata_phi)
iz_norm_phi = get_rescaling(
    train_adata_phi,
    inverse=True
)
print(f"{z_norm_phi=}\n {iz_norm_phi=}")


### Get Transformation from Cell Type Classifier.

In [ ]:
z_norm_g = get_rescaling(train_adata_g)
iz_norm_g = get_rescaling(
    train_adata_g,
    inverse=True
)
print(f"{z_norm_g=}\n {iz_norm_g=}")


## Forward Model.

### Initialize rescaled target prediction model.

In [ ]:
g_model = RescaledTargetPredictionModel(
    target_prediction_model.target_prediction_model,
    inv_params=iz_norm_phi,
    fwd_params=z_norm_g,
)
g_model

### Initialize Forward Model.

In [ ]:
forward_model = ForwardModel(
    forward_model=perturbation_response_prediction_model,
    target_prediction_model=g_model,
)
forward_model

## Predict cell type on full data.

### Forward pass on the model.

In [ ]:
X_train = torch.from_numpy(train_adata_phi.obsm[cell_state_rep]).float().cuda()
X_val = torch.from_numpy(val_adata_phi.obsm[cell_state_rep]).float().cuda()

g_model.eval()
with torch.no_grad():
    G_logits_train = g_model(
        X_train
    )["cell_type"].detach().cpu().numpy()
    G_logits_val = g_model(
        X_val
    )["cell_type"].detach().cpu().numpy()

### Post process output.

In [ ]:
G_probs_train = softmax(G_logits_train, axis=1)
G_probs_val = softmax(G_logits_val, axis=1)

G_id_train = G_probs_train.argmax(1)
G_id_val = G_probs_val.argmax(1)

G_label_train = ct_le.inverse_transform(G_id_train)
G_label_val = ct_le.inverse_transform(G_id_val)
print(f"{G_logits_train.shape=}, {G_logits_val.shape=}, {G_probs_train.shape=}, {G_probs_val.shape=}, {G_id_train.shape=}, {G_id_val.shape=}, {G_label_train.shape=}, {G_label_val.shape=}")


#### Plot proportion of predicted cell types.

In [ ]:
plt.figure(figsize=(5,5))
sns.countplot(np.concatenate((G_label_train, G_label_val), axis=0))
plt.grid(1)

## Posterior Sampling.

### Define settings.

In [ ]:
solver_kwargs_forward_model = {"method": "euler"}
loss_fns = {
"cell_type": lambda pred, target:
    -torch.sum(
        target*torch.nn.functional.log_softmax(pred, dim=-1),
        dim=-1
    )
}


### Initialize guided flow.

In [ ]:
n_fwd_samples = 500
reg = "l1"
reg_strength = 1e-2
n_time_steps_forward_model = 10

guided_flow = LossGuidedFlow(
    prior_flow,
    forward_model,
    loss_fns,
    fix_noise=True,
    num_forward_pass_per_sample=n_fwd_samples,
    regularization=reg,
    reg_strength=reg_strength,
    n_time_steps_forward_model=n_time_steps_forward_model,
    solver_kwargs_forward_model=solver_kwargs_forward_model
)
non_linearity = torch.nn.ReLU()

### Initialize scheduler.

In [ ]:
scheduler_id = "reciprocal"
lmax = 5.0
lmin = 0.0
gamma = 5.5
scheduler = schedulers_dict[scheduler_id](lmax=lmax, lmin=lmin, gamma=gamma)
scheduler

### Visualize schedule.

In [ ]:
# visualizing decaying scheduler
t = torch.linspace(0.0, 1.0, 100)
lambda_t = scheduler.compute_lambda_t(t)
plt.figure(figsize=(3, 3))
plt.plot(t, lambda_t)
plt.title(fr"Decay schedule {scheduler_id}")
plt.xlabel(r"$t$")
plt.ylabel(r"$\lambda_t$")
plt.grid(True)

### Define Target.

In [ ]:
target_cell_type = "MgkPro"
classes = ct_le.classes_.tolist()
n_classes = len(classes)

idx = classes.index(target_cell_type)
target = torch.zeros((n_classes,)).float().to(forward_model.forward_model.device)
target[idx] = 1.0
target = {
    "cell_type": target.unsqueeze(0)
}
target

### Sampling from inverse model.

In [ ]:
use_sde = False
n_samples = 150
n_time_steps = 30 if use_sde else 100
solver_kwargs = {"method": "euler"}
seed = 42
set_reproducibility(seed)

torch.cuda.empty_cache()
traj, loss_history, lambda_history, noise = guided_flow.sample_posterior(
    n_samples,
    optimal_condition=target,
    num_time_steps=n_time_steps,
    solver_kwargs=solver_kwargs,
    lambda_scheduler=scheduler,
    non_linearity=non_linearity,
    sde_sampling=use_sde
)

torch.cuda.empty_cache()
print(f"{traj.shape=}, {loss_history.shape=}, {lambda_history.shape=}, {noise.shape=}")

## Query Forward model with samples. 

### Prepare batch dictionary.

In [ ]:
samples = np.maximum(traj[:, -1, :], 0)
perturbation_reps = next(iter(perturbation_response_prediction_model.train_data.data.perturbation_covariates))
ccondition_data = torch.from_numpy(samples).float().to(perturbation_response_prediction_model.device)
ccondition_dict = {
    perturbation_reps: ccondition_data.unsqueeze(1).repeat(1, n_fwd_samples, 1),
}

batch_dict = {
    DataFields.PERTURBATION_DATA: ccondition_dict,
    DataFields.SOURCE_STATE: noise
}

### Call forward model.

In [ ]:
num_time_steps = 100
cforward_out = forward_model.predict(
    batch_dict,
    return_trajectory=False,
    no_grad=True,
    num_time_steps=num_time_steps,
    solver_kwargs=solver_kwargs,
    fix_noise=True,
)

### Post-process output.

In [ ]:
X_gen = cforward_out[PredictionFields.PREDICTION_DATA].detach().cpu().numpy()
gen_ct_logits = cforward_out[PredictionFields.TARGET_PREDICTION_DATA]["cell_type"].detach().cpu().numpy()

X_channel_gen = X_gen[..., :-n_scatter_feats]
X_scatter_gen = X_gen[..., -n_scatter_feats:]

gen_ct_probs = softmax(gen_ct_logits, axis=-1)
gen_ct_id_label = gen_ct_probs.argmax(-1)
gen_ct_label = ct_le.inverse_transform(gen_ct_id_label.reshape(-1)).\
    reshape(gen_ct_id_label.shape[0], gen_ct_id_label.shape[1])
print(f"{X_gen.shape=}, {X_channel_gen.shape=}, {X_scatter_gen.shape=}, {gen_ct_logits.shape=} {gen_ct_id_label.shape=}, {gen_ct_label.shape=}")


### Plot loss history.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), dpi=50)
t = np.arange(0, 1, 1/loss_history.shape[1])
lines = ax.plot(t, loss_history.T)
fig.suptitle(f"{target_cell_type}")
ax.set_xlabel("Sampling Time.")
ax.set_ylabel("Loss.")
ax.grid(1)
fig.tight_layout(); fig.show()

## Visualize condition covariates.

### Displaying condition samples heatmap.

In [ ]:
emin_loss_idx = np.argmin(loss_history[:, -1])
samples = np.maximum(traj[:, -1, :], 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
fig.suptitle(target_cell_type)
sns.heatmap(samples, annot=False, ax=ax[0], xticklabels=annotation_dict["protocol_columns"])
sns.heatmap(samples[emin_loss_idx][None], annot=True, ax=ax[1], xticklabels=annotation_dict["protocol_columns"], fmt=".2f", annot_kws={"size": 6})

### Hierachical cluster of the samples.

In [ ]:
emin_loss_idx = np.argmin(loss_history[:, -1])
samples = np.maximum(traj[:, -1, :], 0)
sns.clustermap(samples, annot=False, xticklabels=annotation_dict["protocol_columns"], vmin=0.0, vmax=7.0)

## Visualize sampled condition in the learned embedding space.

### Retrieve learned embeddings.

In [ ]:
ccond_emb = perturbation_response_prediction_model.velocity_field.get_condition_embedding(ccondition_dict)[:, 0]
ccond_emb.shape

### Sampling random projection matrices.

In [ ]:
n_projections = 100000
sigma = 0.1
projections = torch.randn((n_projections, ccond_emb.shape[1], 2)).float().cuda() * sigma
print(f"{projections.shape=}")

### Project and average conditions.

In [ ]:
projected_conds_gen = torch.einsum("nd,mdp->nmp", ccond_emb, projections).mean(1).detach().cpu().numpy()
print(f"{projected_conds_gen.shape=}")

### Retrieve unique condition values from train and validation.

In [ ]:
unique_train_conds = np.unique(
    train_adata_phi.obsm["condition_concat"], axis=0
)
unique_val_conds = np.unique(
    val_adata_phi.obsm["condition_concat"], axis=0
)
print(f"{unique_train_conds.shape=}, {unique_val_conds.shape=}")

### Constructing condition dictionaries.

In [ ]:
train_cond_dict = {
    perturbation_reps: torch.from_numpy(unique_train_conds).float().cuda()
}
val_cond_dict = {
    perturbation_reps: torch.from_numpy(unique_val_conds).float().cuda()
}

### Forward pass on condition encoder.

In [ ]:
train_cond_emb = perturbation_response_prediction_model.velocity_field.get_condition_embedding(train_cond_dict)
val_cond_emb = perturbation_response_prediction_model.velocity_field.get_condition_embedding(val_cond_dict)
train_cond_emb.shape, train_cond_emb.shape

### Projecting real conditions.

In [ ]:
projected_conds_train = torch.einsum("nd,mdp->nmp", train_cond_emb, projections).mean(1).detach().cpu().numpy()
projected_conds_val = torch.einsum("nd,mdp->nmp", val_cond_emb, projections).mean(1).detach().cpu().numpy()
print(f"{projected_conds_train.shape=}, {projected_conds_val.shape}")

### Visualize projections.

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.scatter(
    projected_conds_train[:, 0],
    projected_conds_train[:, 1],
    label="train",
    color="grey",
)
ax.scatter(
    projected_conds_val[:, 0],
    projected_conds_val[:, 1],
    label="val",
    color="blue",
)
ax.scatter(
    projected_conds_gen[:, 0],
    projected_conds_gen[:, 1],
    label="val",
    color="red",
)
ax.scatter(
    projected_conds_gen[emin_loss_idx, 0][None],
    projected_conds_gen[emin_loss_idx, 1][None],
    label="val",
    color="green",
)
ax.grid(True)
ax.tick_params("x", rotation=90)

### Compute (L2) distance from real data in embedding space.

In [ ]:
D_espace_learned = cdist(ccond_emb.detach().cpu().numpy(), np.concatenate((train_cond_emb.detach().cpu().numpy(), val_cond_emb.detach().cpu().numpy()), axis=0))
D_espace_learned.shape

### Distribution of mean distance.

In [ ]:
plt.figure(figsize=(5, 3))
sns.histplot(D_espace_learned.mean(0), bins=20, kde=True, stat="density")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xlabel("Mean distance")
plt.ylabel("Density")
plt.title("Distribution of mean(0) of D_espace_learned")
plt.tight_layout()
plt.show()

### Retrieve closest neighbor in embedding space for best sample.

In [ ]:
min_dist_idx = D_espace_learned[emin_loss_idx].argsort()[0].item()
nn_real_cond_best = unique_train_conds[min_dist_idx]

### Heatmap of NN condition.

In [ ]:
emin_loss_idx = np.argmin(loss_history[:, -1])
samples = np.maximum(traj[:, -1, :], 0)
fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
fig.suptitle(f"NN to real data for best sample of {target_cell_type}")
sns.heatmap(nn_real_cond_best[None], annot=True, ax=ax, xticklabels=annotation_dict["protocol_columns"])

## Visualize sampled condition in the original concentration space.

### Sample random projection matrix.

In [ ]:
n_projections = 100000
sigma = 0.1 
projections = torch.randn((n_projections, samples.shape[1], 2)).float().cuda() * sigma
print(f"{projections.shape=}")


### Project and average.

In [ ]:
projected_conds_gen = torch.einsum("nd,mdp->nmp", torch.from_numpy(samples).float().cuda(), projections).mean(1).detach().cpu().numpy()
projected_conds_train = torch.einsum("nd,mdp->nmp", torch.from_numpy(unique_train_conds).float().cuda(), projections).mean(1).detach().cpu().numpy()
projected_conds_val = torch.einsum("nd,mdp->nmp", torch.from_numpy(unique_val_conds).float().cuda(), projections).mean(1).detach().cpu().numpy()
print(f"{projected_conds_train.shape=}, {projected_conds_val.shape}, {projected_conds_gen.shape=}")

### Visualize projections.

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.scatter(
    projected_conds_train[:, 0],
    projected_conds_train[:, 1],
    label="train",
    color="grey",
)
ax.scatter(
    projected_conds_val[:, 0],
    projected_conds_val[:, 1],
    label="val",
    color="blue",
)
ax.scatter(
    projected_conds_gen[:, 0],
    projected_conds_gen[:, 1],
    label="val",
    color="red",
)
ax.scatter(
    projected_conds_gen[emin_loss_idx, 0][None],
    projected_conds_gen[emin_loss_idx, 1][None],
    label="val",
    color="green",
)
ax.grid(True)
ax.tick_params("x", rotation=90)

### Compute L2 distance in original concentration space.

In [ ]:
D_espace_learned = cdist(samples, np.concatenate((unique_train_conds, unique_val_conds), axis=0))
D_espace_learned.shape

### Plot distribution of distances.

In [ ]:
plt.figure(figsize=(5, 3))
sns.histplot(D_espace_learned.mean(0), bins=20, kde=True, stat="density")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xlabel("Mean distance")
plt.ylabel("Density")
plt.title("Distribution of mean(0) of D_espace_learned")
plt.tight_layout()
plt.show()

### Retrieve closest neighbor in embedding space for best sample.

In [ ]:
min_dist_idx = D_espace_learned[emin_loss_idx].argsort()[0].item()
nn_real_cond_best = unique_train_conds[min_dist_idx]

### Heatmap of NN condition.

In [ ]:
emin_loss_idx = np.argmin(loss_history[:, -1])
samples = np.maximum(traj[:, -1, :], 0)
fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
fig.suptitle(f"NN to real data for best sample of {target_cell_type}")
sns.heatmap(nn_real_cond_best[None], annot=True, ax=ax, xticklabels=annotation_dict["protocol_columns"])

## Visualize cell type proportions.

### Displaying induced phenotype heatmap.

In [ ]:
ct_probs = gen_ct_probs.mean(1)
samples = np.maximum(traj[:, -1, :], 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 4), dpi=100)
fig.suptitle(target_cell_type)
sns.heatmap(ct_probs, annot=False, ax=ax[0], xticklabels=ct_le.classes_)
sns.heatmap(ct_probs[emin_loss_idx][None], annot=True, ax=ax[1], xticklabels=ct_le.classes_, fmt=".2f", annot_kws={"size": 6})

## Visualize generated cell states.

### Concatenate real and generated data.

In [ ]:
# concatenate real and generated cell states.
X = np.concatenate(
    (
        X_gen[emin_loss_idx],
        train_adata_phi.obsm[cell_state_rep],
        val_adata_phi.obsm[cell_state_rep]
    ),
    axis=0
)
X_channel = X[:, :-n_scatter_feats]
X_scatter = X[:, -n_scatter_feats:]

# concatenate predicted cell types for real and generated data
G = np.concatenate(
    (
        gen_ct_label[emin_loss_idx],
        G_label_train,
        G_label_val,
    ),
    axis=0
)
print(f"{X_channel.shape=}, {X_scatter.shape=}{G.shape}")

### Create annotated data.

In [ ]:
ct_adata = sc.AnnData(
    X=X_channel,
    obsm={"X_scatter": X_scatter},
    obs={
        "cell_type": G,
        "data_type": ["gen"]*X_gen[emin_loss_idx].shape[0] + \
            ["train"]*len(train_adata_phi) + ["val"]*len(val_adata_phi)
    },
    var=pd.DataFrame(index=train_adata_phi.var_names)
)
sc.pp.pca(ct_adata)
sc.pp.neighbors(ct_adata)
sc.tl.umap(ct_adata)
print(f"{ct_adata=}")


### Visualize UMAP embeddings.

In [ ]:
X_umap = ct_adata.obsm["X_umap"]
ct_mask = (ct_adata.obs["cell_type"] == target_cell_type) & (ct_adata.obs["data_type"] != "gen")
gen_mask = ct_adata.obs["data_type"] == "gen"
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
fig.suptitle(target_cell_type)
axes[0].scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    s=5,
    # alpha=0.2
)
axes[0].scatter(
    X_umap[ct_mask, 0],
    X_umap[ct_mask, 1],
    s=40,
    # alpha=0.9
)
axes[0].set_title(f"Is Target")
axes[0].grid(1)

axes[1].scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    s=5,
    # alpha=0.2
)
axes[1].scatter(
    X_umap[gen_mask, 0],
    X_umap[gen_mask, 1],
    s=40,
    # alpha=0.9
)
axes[1].set_title("Is Generated")
axes[1].grid(1)

for ax in axes:
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")

plt.tight_layout()
plt.show()

### Visualizing expression of (all) generated cells vs real HSCs.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(20, 5))
vmin = 0; vmax = 5.0
sc.pl.dotplot(ct_adata[(ct_adata.obs.data_type != "gen") & (ct_adata.obs.cell_type == target_cell_type)], ct_adata.var_names, "cell_type", ax=ax[0], show=False, vmin=vmin, vmax=vmax)
sc.pl.dotplot(train_adata_g[(train_adata_g.obs.cell_type == target_cell_type)], train_adata_g.var_names, "cell_type", ax=ax[1], show=False, vmin=vmin, vmax=vmax)
sc.pl.dotplot(ct_adata[(ct_adata.obs.data_type == "gen")], ct_adata.var_names, "data_type", ax=ax[2], show=False, vmin=vmin, vmax=vmax)
fig.show()


## Population Distances.

### Computing e-dist between generated samples and cell type subpopulations (Channel Features).

In [ ]:
edist_dict = defaultdict(list)
x_best = X_channel_gen[emin_loss_idx]
for ct in tqdm(classes):
    x_ct = train_adata_g[train_adata_g.obs.cell_type == ct].obsm["X_channel_standardized"]
    edist = compute_e_distance(x_best, x_ct)
    edist_dict["ct"].append(ct)
    edist_dict["edist"].append(edist)
edist_channel_df = pd.DataFrame(edist_dict).sort_values("edist", axis=0, ascending=False)
edist_channel_df

### Plotting values.

In [ ]:
# Create a color column
edist_channel_df["color"] = edist_channel_df["ct"].apply(
    lambda x: "red" if x == target_cell_type else "gray"
)

plt.figure(figsize=(10, 4))
sns.barplot(
    data=edist_channel_df,
    x="ct",
    y="edist",
    palette=edist_channel_df["color"].tolist()
)

plt.grid(True)
plt.xticks(rotation=90)
plt.ylabel("E-distance")
plt.xlabel("Cell type")
plt.title("E-distance per cell type (Channel Features)")
plt.tight_layout()
plt.show()


### Computing e-dist between generated samples and cell type subpopulations (Scatter Features).

In [ ]:
edist_dict = defaultdict(list)
x_best = X_scatter_gen[emin_loss_idx]
for ct in tqdm(classes):
    x_ct = train_adata_g[train_adata_g.obs.cell_type == ct].obsm["X_scatter_standardized"]
    edist = compute_e_distance(x_best, x_ct)
    edist_dict["ct"].append(ct)
    edist_dict["edist"].append(edist)
edist_scatter_df = pd.DataFrame(edist_dict).sort_values("edist", axis=0, ascending=False)
edist_scatter_df

### Plotting values.

In [ ]:
# Create a color column
edist_scatter_df["color"] = edist_scatter_df["ct"].apply(
    lambda x: "red" if x == target_cell_type else "gray"
)

plt.figure(figsize=(10, 4))
sns.barplot(
    data=edist_scatter_df,
    x="ct",
    y="edist",
    palette=edist_scatter_df["color"].tolist()
)

plt.grid(True)
plt.xticks(rotation=90)
plt.ylabel("E-distance")
plt.xlabel("Cell type")
plt.title("E-distance per cell type (Scatter Features)")
plt.tight_layout()
plt.show()


### Plotting cell states induced by nn condition to best sample.

In [ ]:
adata_phi.obs["cell_type"] = np.concatenate(
    (
        G_label_train,
        G_label_val,
    ),
    axis=0
)
X_umap = adata_phi.obsm["X_umap"]
ct_mask = (adata_phi.obs["cell_type"] == target_cell_type)
nn_mask = np.all(adata_phi.obsm["condition_concat"] == nn_real_cond_best, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
fig.suptitle(target_cell_type)
axes[0].scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    s=5,
    # alpha=0.2
)
axes[0].scatter(
    X_umap[ct_mask, 0],
    X_umap[ct_mask, 1],
    s=40,
    # alpha=0.9
)
axes[0].set_title(f"Is Target")
axes[0].grid(1)

axes[1].scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    s=5,
    # alpha=0.2
)
axes[1].scatter(
    X_umap[nn_mask, 0],
    X_umap[nn_mask, 1],
    s=40,
    # alpha=0.9
)
axes[1].set_title("Is NN")
axes[1].grid(1)

for ax in axes:
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")

plt.tight_layout()
plt.show()

## Comparing predictions with real conditions with the highest HSCs proportion.

### Iterate over the unique conditions and retrieve cell type proportions.

In [ ]:
ct_counts_dict = {}
unique_conds =  np.concatenate((unique_train_conds, unique_val_conds))
for idx, cond in enumerate(unique_conds):
    cond_mask = np.all(adata_phi.obsm["condition_concat"] == cond, axis=1)
    cond_adata = adata_phi[cond_mask]
    counts = (cond_adata.obs.cell_type).value_counts().to_dict()
    ct_counts_dict[idx] = counts
ct_counts_df = pd.DataFrame(ct_counts_dict).T.fillna(0)
ct_counts_df = ct_counts_df/np.sum(ct_counts_df.values, axis=1, keepdims=True)
ct_counts_df.head()

### Boxplot of the proportions.

In [ ]:
ct_long = ct_counts_df.reset_index(drop=True).melt(
    var_name="cell_type",
    value_name="proportion"
)

plt.figure(figsize=(12, 5))
sns.boxplot(
    data=ct_long,
    x="cell_type",
    y="proportion"
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("Cell type proportion")
plt.xlabel("Cell type")
plt.tight_layout()
plt.grid(1)
plt.show()


### Retrieving conditioning inducing highest proportion of cell types.

In [ ]:
cond_hsc_max = unique_conds[np.argmax(ct_counts_df.loc[:, "HSCs"])][None]
fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
fig.suptitle(f"Condition with most {target_cell_type}")
sns.heatmap(cond_hsc_max, annot=True, ax=ax, xticklabels=annotation_dict["protocol_columns"])